# FedMMBAF-LIM: Full Runnable Reference Implementation

This notebook implements the main computational ideas in the supplied manuscript:

- **Modality-specific encoders** for face, fingerprint, iris, gait, and voice feature vectors
- **CM-DON**: cross-modal decorrelation regularization
- **SeEvMoG Net**: reliability-aware attention fusion over modality embeddings
- **FeLIM Net**: temporal identity memory with drift-aware prototype updates
- **DP-GC**: gradient clipping, top-k sparsification, low-rank-style compression approximation, and Gaussian noise
- **Federated averaging** across virtual clients, subject-disjoint holdout evaluation, and metrics/plots/CSV outputs

**Important data note:** the manuscript describes SDUMLA-HMT but does not provide the dataset files with this document. The notebook therefore runs end-to-end using a clearly labelled synthetic multimodal demo by default. For real experiments, set `DATA_MODE = "csv"` and supply a feature table with one row per sample/session, a `subject_id` column, optional `session` column, `label` column, and numeric modality feature columns named `face_*`, `fingerprint_*`, `iris_*`, `gait_*`, and `voice_*`. This notebook does not claim that synthetic-demo metrics reproduce the manuscript's reported results. Run the real-data path only with data you are authorized to use.

The notebook is a reproducible reference implementation of the described methods, not a claim of formal differential-privacy certification. DP guarantees depend on accountant choices and end-to-end assumptions.


In [ ]:
# Cell 1 — imports and reproducibility
import os, json, copy, math, random, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, roc_curve, auc
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
# Cell 2 — configuration
# Default is an end-to-end synthetic demonstration. For real extracted features, set DATA_MODE="csv".
DATA_MODE = "synthetic"  # "synthetic" or "csv"
CSV_PATH = "/content/drive/MyDrive/SDUMLA_HMT/features_multimodal.csv"

NUM_CLIENTS = 10
COMMUNICATION_ROUNDS = 5       # Increase to 100 for the manuscript setting (computationally expensive)
LOCAL_EPOCHS = 1               # Increase to 5 for the manuscript setting
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DROPOUT = 0.30
EMBED_DIM = 32
NUM_CLASSES = 2                # Binary verification-style demo; labels encoded from the supplied table
DP_EPSILON_TARGET = 3.0         # Target setting from manuscript; not a certified epsilon accountant
DP_DELTA = 1e-5
CLIP_NORM = 1.0
NOISE_MULTIPLIER = 1.0
GRADIENT_COMPRESSION_RATIO = 0.50
CMDON_LAMBDA = 0.10
DEPENDENCY_THRESHOLD = 0.50
MEMORY_SIZE = 10
DRIFT_THRESHOLD = 0.20
OUTPUT_DIR = Path("./FedMMBAF_LIM_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODALITIES = ["face", "fingerprint", "iris", "gait", "voice"]
print("Outputs:", OUTPUT_DIR.resolve())


In [ ]:
# Cell 3 — data loading / synthetic demonstration
def make_synthetic_multimodal(n_subjects=106, sessions=5, samples_per_session=2, seed=SEED):
    """Synthetic data only for testing the pipeline; not SDUMLA-HMT data."""
    rng = np.random.default_rng(seed)
    rows = []
    # Subject labels emulate enrolled identity groups, with mild session drift.
    for sid in range(n_subjects):
        label = sid % 2
        latent = rng.normal(0, 1, 12) + (label * 0.8)
        for sess in range(sessions):
            for _ in range(samples_per_session):
                row = {"subject_id": f"S{sid:03d}", "session": sess, "label": label}
                for mod, dim in zip(MODALITIES, [24, 20, 16, 18, 22]):
                    projection = rng.normal(0, 0.45, (12, dim))
                    drift = rng.normal(0, 0.03 * (sess + 1), dim)
                    x = latent @ projection + drift + rng.normal(0, 0.55, dim)
                    # Randomly simulate absent/degraded modalities.
                    if rng.random() < 0.06:
                        x = np.zeros_like(x)
                    elif rng.random() < 0.08:
                        x = x + rng.normal(0, 1.5, dim)
                    for j, v in enumerate(x):
                        row[f"{mod}_{j:03d}"] = float(v)
                rows.append(row)
    return pd.DataFrame(rows)

def load_feature_table():
    if DATA_MODE == "synthetic":
        df = make_synthetic_multimodal()
        print("Using SYNTHETIC demonstration data; not real biometric data.")
    elif DATA_MODE == "csv":
        if not os.path.isfile(CSV_PATH):
            raise FileNotFoundError(
                f"CSV not found: {CSV_PATH}\nSet CSV_PATH to your extracted multimodal feature CSV."
            )
        df = pd.read_csv(CSV_PATH)
    else:
        raise ValueError("DATA_MODE must be 'synthetic' or 'csv'.")

    required = {"subject_id", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV must include columns: {sorted(required)}; missing {sorted(missing)}")
    if "session" not in df.columns:
        df["session"] = 0

    modality_cols = {}
    for mod in MODALITIES:
        cols = [c for c in df.columns if c.lower().startswith(mod.lower() + "_") and pd.api.types.is_numeric_dtype(df[c])]
        modality_cols[mod] = cols
    absent = [m for m, cols in modality_cols.items() if not cols]
    if absent:
        raise ValueError(
            f"No numeric feature columns found for modalities {absent}. Expected prefixes like face_000, "
            "fingerprint_000, iris_000, gait_000, voice_000. Extract modality features first."
        )

    # Convert labels to integer classes.
    le = LabelEncoder()
    df["label_encoded"] = le.fit_transform(df["label"].astype(str))
    if df["label_encoded"].nunique() < 2:
        raise ValueError("At least two distinct labels are required.")
    # Impute numeric feature columns with training-independent medians only for basic usability.
    # For rigorous experiments, fit imputation/scaling on training subjects only.
    for cols in modality_cols.values():
        df[cols] = df[cols].replace([np.inf, -np.inf], np.nan)
        df[cols] = df[cols].fillna(df[cols].median()).fillna(0.0)
    return df, modality_cols, le

df, MODALITY_COLS, LABEL_ENCODER = load_feature_table()
print("Rows:", len(df), "| subjects:", df.subject_id.nunique(), "| labels:", list(LABEL_ENCODER.classes_))
print("Modality dimensions:", {m: len(c) for m,c in MODALITY_COLS.items()})
display(df.head(3))


In [ ]:
# Cell 4 — subject-independent train/test split and modality standardization
# GroupShuffleSplit keeps each subject wholly in either train or test.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, df["label_encoded"], groups=df["subject_id"]))
train_df = df.iloc[train_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

overlap = set(train_df.subject_id) & set(test_df.subject_id)
assert not overlap, f"Subject leakage detected: {len(overlap)} subjects"

SCALERS = {}
for mod, cols in MODALITY_COLS.items():
    sc = StandardScaler()
    train_df.loc[:, cols] = sc.fit_transform(train_df[cols].astype(float))
    test_df.loc[:, cols] = sc.transform(test_df[cols].astype(float))
    SCALERS[mod] = sc

NUM_CLASSES = int(df["label_encoded"].nunique())
print(f"Train rows={len(train_df)}, test rows={len(test_df)}, held-out subjects={test_df.subject_id.nunique()}")
print("Classes:", NUM_CLASSES)


In [ ]:
# Cell 5 — dataset and non-IID virtual client partition
class MultimodalFeatureDataset(Dataset):
    def __init__(self, frame, modality_cols):
        self.frame = frame.reset_index(drop=True)
        self.modality_cols = modality_cols
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        xs = {m: torch.tensor(row[cols].to_numpy(dtype=np.float32), dtype=torch.float32)
              for m, cols in self.modality_cols.items()}
        y = torch.tensor(int(row["label_encoded"]), dtype=torch.long)
        return xs, y

train_dataset = MultimodalFeatureDataset(train_df, MODALITY_COLS)
test_dataset = MultimodalFeatureDataset(test_df, MODALITY_COLS)

def make_client_indices(frame, n_clients=NUM_CLIENTS, seed=SEED):
    """Partition training rows into virtual clients with label-skewed/non-IID shards."""
    rng = np.random.default_rng(seed)
    indices = np.arange(len(frame))
    labels = frame["label_encoded"].to_numpy()
    # Sort by label, split into shards, then distribute shards to clients.
    sorted_idx = indices[np.argsort(labels)]
    shards = np.array_split(sorted_idx, n_clients * 2)
    rng.shuffle(shards)
    client_shards = [[] for _ in range(n_clients)]
    for i, shard in enumerate(shards):
        client_shards[i % n_clients].extend(shard.tolist())
    clients = [np.asarray(x, dtype=int) for x in client_shards if len(x)]
    return clients

CLIENT_INDICES = make_client_indices(train_df)
print("Virtual clients:", len(CLIENT_INDICES), "| client sample counts:", [len(x) for x in CLIENT_INDICES])


In [ ]:
# Cell 6 — FedMMBAF-LIM model modules
class ModalityEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=EMBED_DIM, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, embed_dim), nn.LayerNorm(embed_dim)
        )
        self.reliability = nn.Sequential(nn.Linear(embed_dim, 1), nn.Sigmoid())
    def forward(self, x):
        z = self.net(x)
        confidence = self.reliability(z)
        return z, confidence

class FedMMBAFLIM(nn.Module):
    def __init__(self, modality_dims, n_classes):
        super().__init__()
        self.modalities = list(modality_dims.keys())
        self.encoders = nn.ModuleDict({
            m: ModalityEncoder(modality_dims[m]) for m in self.modalities
        })
        # SeEvMoG-style reliability-aware attention across modality nodes.
        self.node_score = nn.Sequential(nn.Linear(EMBED_DIM + 1, 32), nn.Tanh(), nn.Linear(32, 1))
        self.fusion = nn.Sequential(
            nn.Linear(EMBED_DIM, 64), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(64, n_classes)
        )
    def forward(self, xs):
        embeddings, confs = [], []
        for m in self.modalities:
            z, c = self.encoders[m](xs[m])
            embeddings.append(z)
            confs.append(c)
        Z = torch.stack(embeddings, dim=1)       # [B, modalities, embed]
        C = torch.stack(confs, dim=1)            # [B, modalities, 1]
        scores = self.node_score(torch.cat([Z, C], dim=-1)).squeeze(-1)
        # Reliability-aware weighting: low confidence decreases attention.
        scores = scores + torch.log(C.squeeze(-1).clamp_min(1e-6))
        attn = torch.softmax(scores, dim=1)
        fused = torch.sum(Z * attn.unsqueeze(-1), dim=1)
        logits = self.fusion(fused)
        return logits, Z, C, attn, fused

def cmdon_decorrelation_loss(Z):
    """Penalize high cross-modal correlation while retaining classification loss."""
    # Z: [batch, modalities, embed]
    if Z.shape[0] < 2: return Z.new_tensor(0.0)
    Zc = Z - Z.mean(dim=0, keepdim=True)
    Zc = F.normalize(Zc, dim=0)
    losses = []
    for i in range(Z.shape[1]):
        for j in range(i+1, Z.shape[1]):
            corr = torch.mean(torch.sum(Zc[:, i, :] * Zc[:, j, :], dim=-1).abs())
            # Thresholded dependency suppression (soft penalty above threshold).
            losses.append(F.relu(corr - DEPENDENCY_THRESHOLD))
    return torch.stack(losses).mean() if losses else Z.new_tensor(0.0)

modality_dims = {m: len(MODALITY_COLS[m]) for m in MODALITIES}
global_model = FedMMBAFLIM(modality_dims, NUM_CLASSES).to(DEVICE)
print(global_model)


In [ ]:
# Cell 7 — DP-GC gradient transform and local client training
def dp_gc_gradients(model, clip_norm=CLIP_NORM, noise_multiplier=NOISE_MULTIPLIER,
                    keep_ratio=GRADIENT_COMPRESSION_RATIO):
    """Practical gradient clipping + top-k sparsification + Gaussian noise.
    This is a reference mechanism, not a formal DP accountant implementation.
    """
    params = [p for p in model.parameters() if p.grad is not None]
    if not params: return
    torch.nn.utils.clip_grad_norm_(params, clip_norm)
    for p in params:
        g = p.grad
        flat = g.view(-1)
        k = max(1, int(flat.numel() * keep_ratio))
        if k < flat.numel():
            _, idx = torch.topk(flat.abs(), k, sorted=False)
            mask = torch.zeros_like(flat)
            mask[idx] = 1.0
            flat = flat * mask
        # Gaussian perturbation after clipping/sparsification.
        flat = flat + torch.randn_like(flat) * (noise_multiplier * clip_norm / max(1, len(params)))
        p.grad.copy_(flat.view_as(g))

def batch_to_device(xs, y):
    return {k:v.to(DEVICE) for k,v in xs.items()}, y.to(DEVICE)

def train_local(global_state, indices, epochs=LOCAL_EPOCHS):
    model = FedMMBAFLIM(modality_dims, NUM_CLASSES).to(DEVICE)
    model.load_state_dict(copy.deepcopy(global_state))
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    subset = torch.utils.data.Subset(train_dataset, indices.tolist())
    loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    model.train()
    losses = []
    for _ in range(epochs):
        for xs, y in loader:
            xs, y = batch_to_device(xs, y)
            opt.zero_grad(set_to_none=True)
            logits, Z, C, attn, fused = model(xs)
            cls_loss = F.cross_entropy(logits, y)
            loss = cls_loss + CMDON_LAMBDA * cmdon_decorrelation_loss(Z)
            loss.backward()
            dp_gc_gradients(model)
            opt.step()
            losses.append(float(loss.detach().cpu()))
    return copy.deepcopy(model.state_dict()), len(indices), float(np.mean(losses) if losses else np.nan)

def fedavg(states, sizes):
    total = float(sum(sizes))
    out = copy.deepcopy(states[0])
    for key in out:
        if torch.is_floating_point(out[key]):
            out[key] = sum(states[i][key] * (sizes[i] / total) for i in range(len(states)))
        else:
            out[key] = states[0][key]
    return out


In [ ]:
# Cell 8 — federated training loop
global_state = copy.deepcopy(global_model.state_dict())
history = []
for rnd in range(COMMUNICATION_ROUNDS):
    local_states, local_sizes, local_losses = [], [], []
    for client_id, idx in enumerate(CLIENT_INDICES):
        state, n, loss = train_local(global_state, idx)
        local_states.append(state); local_sizes.append(n); local_losses.append(loss)
    global_state = fedavg(local_states, local_sizes)
    global_model.load_state_dict(global_state)
    history.append({"round": rnd + 1, "mean_local_loss": float(np.nanmean(local_losses))})
    print(f"Round {rnd+1:03d}/{COMMUNICATION_ROUNDS} | mean local loss={history[-1]['mean_local_loss']:.4f}")

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "federated_training_history.csv", index=False)


In [ ]:
# Cell 9 — FeLIM temporal identity memory and test evaluation
class FeLIMMemory:
    """Per-client-style prototype bank with drift-sensitive selective updates."""
    def __init__(self, max_prototypes=MEMORY_SIZE, drift_threshold=DRIFT_THRESHOLD):
        self.max_prototypes = max_prototypes
        self.drift_threshold = drift_threshold
        self.prototypes = {}  # subject_id -> list of numpy embeddings
    def update(self, subject_id, embedding):
        e = np.asarray(embedding, dtype=np.float32)
        bank = self.prototypes.setdefault(str(subject_id), [])
        if not bank:
            bank.append(e); return
        sims = [float(np.dot(e, p) / (np.linalg.norm(e)*np.linalg.norm(p) + 1e-8)) for p in bank]
        drift = 1.0 - max(sims)
        if drift >= self.drift_threshold:
            bank.append(e)
            if len(bank) > self.max_prototypes: bank.pop(0)
        else:
            # Smooth update of the closest prototype to preserve identity consistency.
            j = int(np.argmax(sims))
            bank[j] = 0.9 * bank[j] + 0.1 * e

@torch.no_grad()
def evaluate_model(model, frame, dataset, update_memory=False):
    model.eval()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    y_true, y_pred, y_prob = [], [], []
    all_embeddings = []
    for xs, y in loader:
        xs, y = batch_to_device(xs, y)
        logits, Z, C, attn, fused = model(xs)
        prob = torch.softmax(logits, dim=1)
        pred = prob.argmax(dim=1)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())
        y_prob.extend(prob.cpu().numpy().tolist())
        all_embeddings.extend(fused.cpu().numpy())
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); y_prob = np.asarray(y_prob)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if NUM_CLASSES == 2:
        metrics["specificity"] = (confusion_matrix(y_true, y_pred, labels=[0,1])[0,0] /
                                  max(1, confusion_matrix(y_true, y_pred, labels=[0,1])[0].sum()))
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_prob[:,1])
            metrics["average_precision"] = average_precision_score(y_true, y_prob[:,1])
        except ValueError:
            metrics["roc_auc"] = np.nan; metrics["average_precision"] = np.nan
    if update_memory:
        memory = FeLIMMemory()
        for i, e in enumerate(all_embeddings):
            memory.update(frame.iloc[i]["subject_id"], e)
        print("FeLIM memory subjects:", len(memory.prototypes))
    return metrics, y_true, y_pred, y_prob, np.asarray(all_embeddings)

test_metrics, y_true, y_pred, y_prob, test_embeddings = evaluate_model(
    global_model, test_df, test_dataset, update_memory=True
)
print("\nHeld-out subject test metrics")
display(pd.DataFrame([test_metrics]).T.rename(columns={0:"value"}))
print(classification_report(y_true, y_pred, zero_division=0))
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)


In [ ]:
# Cell 10 — confusion matrix, ROC, training curve, and metric exports
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("FedMMBAF-LIM Confusion Matrix", fontsize=15, fontweight="bold")
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=12, fontweight="bold")
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

if NUM_CLASSES == 2 and len(np.unique(y_true)) == 2:
    fpr, tpr, _ = roc_curve(y_true, y_prob[:,1])
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(7,6))
    ax.plot(fpr, tpr, linewidth=2, label=f"ROC AUC = {roc_auc:.4f}")
    ax.plot([0,1], [0,1], linestyle="--")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve", fontsize=15, fontweight="bold")
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "roc_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

fig, ax = plt.subplots(figsize=(7,5))
ax.plot(history_df["round"], history_df["mean_local_loss"], marker="o")
ax.set_xlabel("Communication round"); ax.set_ylabel("Mean local loss")
ax.set_title("Federated Training Loss", fontsize=15, fontweight="bold")
ax.grid(alpha=.25); fig.tight_layout()
fig.savefig(OUTPUT_DIR / "federated_loss.png", dpi=300, bbox_inches="tight")
plt.show()

pd.DataFrame(cm).to_csv(OUTPUT_DIR / "confusion_matrix.csv", index=False)
pred_df = test_df[["subject_id", "session", "label"]].copy()
pred_df["predicted_label_encoded"] = y_pred
pred_df["predicted_label"] = LABEL_ENCODER.inverse_transform(y_pred)
for k in range(y_prob.shape[1]):
    pred_df[f"prob_class_{k}"] = y_prob[:,k]
pred_df.to_csv(OUTPUT_DIR / "heldout_predictions.csv", index=False)


In [ ]:
# Cell 11 — manuscript reference tables (reported values, not recomputed results)
# These are transcribed reference values for plotting/table reproduction only.
# They are NOT metrics computed by the model in this notebook.
reported_baselines = pd.DataFrame([
    ["FedAvg", 92.84, 91.76, 91.21, 93.18, 91.48],
    ["FedProx", 93.47, 92.61, 92.08, 93.76, 92.34],
    ["SCAFFOLD", 94.12, 93.27, 92.74, 94.51, 93.00],
    ["FedAdam", 94.68, 93.91, 93.46, 95.02, 93.68],
    ["FedAvg + DP", 95.31, 94.48, 94.02, 95.77, 94.25],
    ["FedAvg + Secure Aggregation", 95.76, 94.93, 94.51, 96.18, 94.71],
    ["FedMMBAF-LIM (reported)", 98.76, 98.42, 97.95, 98.88, 98.18],
], columns=["Method","Accuracy (%)","Precision (%)","Sensitivity (%)","Specificity (%)","F1-score (%)"])
reported_baselines.to_csv(OUTPUT_DIR / "manuscript_reported_baselines.csv", index=False)
display(reported_baselines)

fig, ax = plt.subplots(figsize=(12,6))
x = np.arange(len(reported_baselines))
ax.bar(x, reported_baselines["Accuracy (%)"])
ax.set_xticks(x); ax.set_xticklabels(reported_baselines["Method"], rotation=25, ha="right")
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(85,100)
ax.set_title("Manuscript-Reported Baseline Accuracy (reference only)", fontsize=15, fontweight="bold")
ax.grid(axis="y", alpha=.25); fig.tight_layout()
fig.savefig(OUTPUT_DIR / "reported_baseline_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 12 — save model, scalers, label encoder, config, and ZIP all outputs
torch.save({
    "model_state_dict": global_model.state_dict(),
    "modality_dims": modality_dims,
    "num_classes": NUM_CLASSES,
    "modalities": MODALITIES,
    "label_classes": LABEL_ENCODER.classes_.tolist(),
    "config": {
        "num_clients": NUM_CLIENTS,
        "communication_rounds": COMMUNICATION_ROUNDS,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "cmdon_lambda": CMDON_LAMBDA,
        "dependency_threshold": DEPENDENCY_THRESHOLD,
        "memory_size": MEMORY_SIZE,
        "drift_threshold": DRIFT_THRESHOLD,
        "clip_norm": CLIP_NORM,
        "noise_multiplier": NOISE_MULTIPLIER,
        "gradient_compression_ratio": GRADIENT_COMPRESSION_RATIO
    }
}, OUTPUT_DIR / "fedmmbaf_lim_model.pt")

with open(OUTPUT_DIR / "run_config.json", "w") as f:
    json.dump({
        "data_mode": DATA_MODE,
        "synthetic_demo": DATA_MODE == "synthetic",
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "train_subjects": int(train_df.subject_id.nunique()),
        "test_subjects": int(test_df.subject_id.nunique()),
        "test_metrics": {k: (float(v) if np.isfinite(v) else None) for k,v in test_metrics.items()}
    }, f, indent=2)

zip_path = Path("FedMMBAF_LIM_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(Path(OUTPUT_DIR.name) / p.relative_to(OUTPUT_DIR)))
print("Saved model and outputs.")
print("Output folder:", OUTPUT_DIR.resolve())
print("ZIP:", zip_path.resolve())


## Real SDUMLA-HMT data integration checklist

1. Download/access SDUMLA-HMT through the dataset's authorized distribution channel and keep raw biometric files local.
2. Extract modality-specific embeddings/features into a single CSV with columns:
   - `subject_id`, `session`, `label`
   - `face_000 ...`, `fingerprint_000 ...`, `iris_000 ...`, `gait_000 ...`, `voice_000 ...`
3. Change `DATA_MODE = "csv"` and update `CSV_PATH`.
4. For a paper-quality evaluation, use a fixed subject-independent split shared by every baseline, fit all preprocessing only on training subjects, report multiple random seeds, and use a proper accountant (e.g. Rényi DP accountant) if claiming a formal epsilon bound.
5. If the dataset has no voice modality, do not fabricate it for real experiments: adapt `MODALITIES` and the model to the actually available modalities, and clearly document the difference from the manuscript setup.
